**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/costabile-2023"], check=True)
    os.chdir("/content/encore/entries/costabile-2023")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Costabile et al. (2023) from dandisets 000235, 000236, 000237 and 000238

**Paper.** Costabile JD, Balakrishnan KA, Schwinn S, Haesemeyer M (2023). Model discovery to link
neural activity to behavioral tasks. *eLife* 12:e83289.
[doi:10.7554/eLife.83289](https://doi.org/10.7554/eLife.83289)

**Datasets.** DANDI:000235 "Thermoregulatory Responses Forebrain" version 0.230316.1600,
DANDI:000236 "Midbrain" version 0.230316.2031, DANDI:000237 "Hindbrain" version 0.230316.1655 and
DANDI:000238 "Thermoregulatory Responses Reticulospinal system" version 0.230316.1519. The paper
cites no version (its data list names the dandisets only). Each dandiset has a single published
version, all made on 2023-03-16, and the draft of each has the same asset count and size; the
published versions are pinned because they are the only fixed ones and the most recent.

**Claims.** `claims/costabile-2023.yaml`, written from the version of record (eLife, 6 June 2023)
and checked by a second reader. The paper's code is github.com/haesemeyer/mine_pub at commit
368d756; the tail-processing code it names is bitbucket.org/jcostabile/2p, read here at commit
5ffaa2d.

**What the dandisets hold.** Per fish, one NWB file with, for each of 30 planes, the raw
two-channel movie, CaImAn's segmentation and inferred calcium, the temperature stimulus and the
250 Hz tail tracking. They hold no fitted models, no Taylor metrics, no swim-bout calls, no
registration to the Z-Brain atlas, and no annotation of the backfilled reticulospinal neurons.

**Scope.** Most claims rest on the MINE CNN fit to every one of 432,882 neurons (and again to
rotated control data), with Taylor bootstraps on the fit ones. Section 5 fits the paper's linear
comparison model to every bright component and times the CNN on this machine; the full set of CNN
fits is far beyond the run limit, so those claims are blocked `too_large`. A second notebook,
`mine_fits_subsample.ipynb`, fits the CNN to a random subsample of 600 components and estimates
MINE's side of fig6d-mine-identifies-more-than-lm and figs6-mine-more-at-every-cutoff; the estimates
decide no outcome. The simulation claims (Figures 1 to 4) and the mouse claims (Figure 5) use data
that are not in these dandisets (`data: elsewhere`); they were not attempted in this run (rough
costs in section 5).

| Claim | Tested here |
|---|---|
| data-zebrafish-coverage | tested (section 3) |
| figs6-train-test-uncorrelated | tested (section 4) |
| fig6d, figs6-shuffle-enrichment, figs6-mine-more-at-every-cutoff, fig6e (two), fig6g, fig7b, fig7d, fig7e, lm-recovers-fewer-clusters, temperature-nonlinear-behavior-linear, interaction-terms-small | blocked, too_large (section 5; estimates for fig6d and figs6-mine-more-at-every-cutoff in `mine_fits_subsample.ipynb`) |
| fig6f-reticulospinal-behavior | blocked, data_missing: the backfill annotation is not in 000238 (section 2); it would also need the fits |
| figs6 region claims (three), fig7a, fig7c, fig7f | blocked, too_large through their dependence on the fits; they also need the Z-Brain registration, which is not in the dandisets |
| Figures 1 to 5 (simulations, mouse widefield; data elsewhere) | blocked, other: not attempted in this run |


In [ ]:
import os, time, resource, zlib
os.environ.setdefault("OMP_NUM_THREADS", "1")
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mine_data as md

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
T0 = time.time()

## 1. Data check

The archive's metadata for each pinned version, and every asset.

In [ ]:
assets = {}
for d in md.VERSIONS:
    meta = md.version_metadata(d)
    assets[d] = md.list_assets(d)
    s = meta["assetsSummary"]
    print(f"{d} {md.VERSIONS[d]}: {meta['name']!r}; {s['numberOfFiles']} files, {s['numberOfSubjects']} subjects, "
          f"{s['numberOfBytes']/1e9:.1f} GB")
    print("   description:", meta["description"])
    for a in assets[d]:
        print("   ", a["path"], f"{a['size']/1e9:.2f} GB")

One file, listed in full: every group and dataset of the first plane (the other 29 planes have
the same structure), and the counts of each kind of object.

In [ ]:
import h5py
f = md.open_nwb(assets["000235"][0]["asset_id"])
kinds = {}
def visit(name, obj):
    parts = name.split("/")
    if any(p.endswith("_Z0") or p in ("Z0_0", "Z0_1", "Segmentation_Z0") for p in parts) or (len(parts) <= 1 and not parts[0].startswith(("Z", "Ca", "Dff"))):
        if isinstance(obj, h5py.Dataset):
            print(f"  {name} {obj.shape} {obj.dtype} {obj.attrs.get('unit', '')}")
        else:
            print(f"  {name}/ {obj.attrs.get('neurodata_type', '')}")
    if isinstance(obj, h5py.Group):
        t = obj.attrs.get("neurodata_type", "")
        t = t.decode() if isinstance(t, bytes) else t
        kinds[t] = kinds.get(t, 0) + 1
for top in ["acquisition", "processing", "stimulus", "general", "scratch"]:
    if isinstance(f[top], h5py.Group):
        f[top].visititems(visit)
print({k: v for k, v in kinds.items() if k})
print("reference_images in Segmentation_Z0:", list(f["processing/ophys/ImageSegmentation/Segmentation_Z0/reference_images"].keys()))
print("experiment_description:", f["general/experiment_description"][()].decode()[:600])
for k in ("Z0_0/Ch0", "Z0_1/Ch1"):
    g = f[f"general/optophysiology/{k}"]
    print(f"{k}: description {g['description'][()]!r}, emission {g['emission_lambda'][()]} nm; plane indicator {f['general/optophysiology/' + k.split('/')[0] + '/indicator'][()]!r}")
f.close()


### One pass over every plane

Each plane is read once and reduced as it loads (`mine_data.plane_pass`): counts and shapes, the
temperature trace on the 5 Hz base, the tail metrics and bouts under each bout reading, the
brightness of every component (the mean of the GCaMP movie over imaging time, over the
component's pixels), and the lagged correlations of the calcium traces with temperature and with
swim starts for the clock check. The movie is read for the 25 whole-brain fish only; the backfill
files are summarised without it. Four processes run in parallel.

In [ ]:
SUBSET = os.environ.get("COSTABILE_SUBSET")  # development only: first file of each dandiset, 3 planes
tasks = []
for d in md.VERSIONS:
    for a in (assets[d][:1] if SUBSET else assets[d]):
        with md.open_nwb(a["asset_id"]) as f:
            pl = md.planes(f)
        for p in (pl[4:7] if SUBSET else pl):
            tasks.append((a["asset_id"], d, p, d in md.ZEBRAFISH_WHOLE_BRAIN))
print(len(tasks), "planes in", sum(len(v) for v in assets.values()), "files")
t = time.time()
with ProcessPoolExecutor(4) as ex:
    res = list(ex.map(md.plane_pass, *zip(*tasks), chunksize=2))
print(f"pass took {time.time()-t:.0f} s")
pp = pd.DataFrame([{k: v for k, v in r.items() if not isinstance(v, (np.ndarray, dict))} for r in res])
pp["fish_short"] = pp.fish.str.split("_").str[0].str.replace("sub-", "")
print("peak memory of this process (GB):", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6,
      "; of the workers (GB, largest):", resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1e6)

In [ ]:
fish = pp.groupby(["dandiset", "fish"]).agg(planes=("plane", "size"), plane_min=("plane", "min"), plane_max=("plane", "max"),
    rois=("n_rois", "sum"), frames_min=("n_frames", "min"), frames_max=("n_frames", "max"),
    ca_rate=("ca_rate", "first"), frame_s=("frame_duration", "first"), temp_min=("temp_min", "min"),
    temp_max=("temp_max", "max"), tail_dt_ms=("tail_dt", lambda x: 1000 * np.median(x))).reset_index()
print(fish.to_string())
print()
print("files per dandiset:", fish.groupby("dandiset").size().to_dict())
print("planes per file:", sorted(fish.planes.unique()), " frames per plane:", pp.n_frames.min(), "to", pp.n_frames.max())
print("imaging rate (Hz):", sorted(pp.ca_rate.unique()), " = 1 / frame duration", sorted(pp.frame_duration.unique()),
      " -> 1/0.4194304 =", 1 / 0.4194304)
print("plane length (s):", (pp.n_frames * pp.frame_duration).min(), "to", (pp.n_frames * pp.frame_duration).max())
print("temperature range over all planes (C):", pp.temp_min.min(), "to", pp.temp_max.max(),
      "; samples per plane", pp.temp_n.min(), "to", pp.temp_n.max(), "at 20 Hz =", pp.temp_n.min() / 20, "to", pp.temp_n.max() / 20, "s")
print("tail sampling interval (ms), median per plane:", round(1000 * pp.tail_dt.min(), 3), "to", round(1000 * pp.tail_dt.max(), 3),
      "; tail time range (s):", round(pp.tail_t0.min(), 2), "to", round(pp.tail_t1.max(), 2))
print("cumulative tail angle equals the sum of the segment angles: max abs difference", pp.cum_matches_sum.max())
print("ROI ids are 0..n-1 in every plane:", pp.rois_ok.all(), "; masks per plane equal traces per plane:", (pp.n_rois == pp.n_rois_masks).all())
print("mask coordinates within the 512 x 512 frame:", pp.mask_x_max.max(), pp.mask_y_max.max(), "; movie shapes:", pp.movie_shape.astype(str).unique())
print("calcium traces: NaN", pp.ca_nan.sum(), "; range", pp.ca_min.min(), "to", pp.ca_max.max())
print("tail segment angles: NaN values", pp.tail_nan.sum(), "in", (pp.tail_nan > 0).sum(), "planes")

Checks of the description against the values:

- 25 whole-brain fish (8 + 9 + 8 files), and 6 backfill fish in 000238, as the paper states.
  Every whole-brain file has 30 planes, the paper's "30 planes ... in each fish"; one backfill
  file (RSpinal-Random-Wave-01) has 25 (section 2). A few planes have 1181 frames, not 1182.
- The imaging rate is 2.384 Hz in every file, the reciprocal of the frame duration of 0.4194304 s
  in the experiment description. The Methods say 2.5 Hz. The plane length, 1182 frames of
  0.419 s (495.8 s), matches the paper's 495 s of stimulus per plane, so 2.384 Hz is the
  acquisition rate and the paper's 2.5 Hz is rounded (a finding, minor).
- Pixel size 0.78125 um (the paper's 0.78 um) and the 512 x 512 frame.
- Temperature is stored in degrees C at 20 Hz and starts at 22 C, the baseline the code adds to
  the temperature model's output, so it is the converted trace. Its maximum exceeds the paper's
  30 C in some planes, and each trace is up to about 11 s longer than the plane's imaging (section 2).
- The tail is sampled every 4.0 ms (250 Hz). In most planes the tail time runs from a few seconds
  before the first imaging frame to its end, so tail time 0 is the start of imaging (as the code
  treats it); the planes that differ are listed in section 2.
- The cumulative tail angle is exactly the sum of the segment angles, so the segment angles are
  relative angles between tail segments.
- The pixel masks' `x` is the column and `y` the row: a trace averaged over the mask pixels of the
  movie follows the CaImAn trace only in that orientation (next cell).

In [ ]:
# Orientation of the pixel masks: correlate each of the 40 brightest components' CaImAn trace with
# the movie averaged over its mask pixels, read as (row=y, col=x) and as (row=x, col=y).
from scipy.ndimage import uniform_filter1d
a = assets["000235"][0]["asset_id"]
with md.open_nwb(a) as f:
    mov = f["acquisition/Z5_0/data"][:].astype(np.float32)
    pm, st, idx = md.masks(f, 5)
    C = f["processing/ophys/CaUnitRespones_Z5/data"][:]
proj = mov.mean(0)
br = md.component_brightness(pm, st, idx, proj)
top = np.argsort(br)[::-1][:40]
for name, (r_, c_) in {"row=y, col=x": ("y", "x"), "row=x, col=y": ("x", "y")}.items():
    rs = [np.corrcoef(uniform_filter1d(mov[:, pm[r_][st[j]:idx[j]], pm[c_][st[j]:idx[j]]].mean(1), 3), C[:, j])[0, 1] for j in top]
    print(f"{name}: median r = {np.nanmedian(rs):.3f} (range {np.nanmin(rs):.3f} to {np.nanmax(rs):.3f})")
del mov

### Recordings deposited more than once

Each plane's temperature trace is a random wave of its own, and its calcium and tail traces are
its own. A plane deposited twice (within a file or across files and dandisets) would repeat a
digest, or have a temperature trace that correlates almost perfectly with another plane's.

In [ ]:
for col in ["digest_ca", "digest_temp", "digest_tail"]:
    dup = pp[pp.duplicated(col, keep=False)]
    print(col, ": repeated digests", dup[col].nunique(), "over", len(dup), "planes")
T5 = np.vstack([r["temp5"][:2300] for r in res])
Tz = (T5 - T5.mean(1, keepdims=True)) / (T5.std(1, keepdims=True) + 1e-9)
R = Tz @ Tz.T / Tz.shape[1]
np.fill_diagonal(R, np.nan)
iu = np.triu_indices_from(R, 1)
print(f"temperature traces of different planes: largest correlation {np.nanmax(R):.3f}; "
      f"pairs above 0.99: {(R[iu] > 0.99).sum()} of {iu[0].size}; 99.9th percentile {np.nanpercentile(R[iu], 99.9):.3f}")

### Do the calcium, stimulus and tail data share a clock?

For each plane of the 25 whole-brain fish, the components that pass the brightness cut are put on
the 5 Hz base, z-scored and smoothed as the code does. Two checks:

- Temperature, by decoding with the labels shifted in time: a Ridge model of each component on
  the temperature over the preceding 10 s (as the paper's linear comparison model, temperature
  only) is fit on the first two-thirds and scored on the last third, with the temperature trace
  displaced by -30 to +30 s. With a shared clock the held-out correlation peaks at no displacement.
  The temperature of the next plane of the same fish, with the same statistics, is the mismatch
  control. Raw lagged correlations are printed too; for a slow stimulus they are high by chance,
  so they do not separate a response from an offset.
- Swim starts (bout reading `thr5_min10`): the lagged correlation of calcium with the bout-start
  count; motor-related activity should rise in the seconds after a bout and not before.

Each curve is the mean over planes of the mean of the top 5% of components.

In [ ]:
ok = [r for r in res if "clock_temp_top" in r]
lags = ok[0]["clock_lags_s"]
own = np.vstack([r["clock_temp_top"] for r in ok]); oth = np.vstack([r["clock_temp_other_top"] for r in ok])
best_lag = lags[np.argmax(own, 1)]
print(f"{len(ok)} planes with at least 5 bright components")
print("mean top-5% |r| with own temperature by lag (s):")
print(pd.Series(own.mean(0).round(3), index=lags).iloc[::5].to_string())
print(f"peak of the mean curve at {lags[np.argmax(own.mean(0))]:.0f} s; per-plane peak lag median {np.median(best_lag):.0f} s, "
      f"25th to 75th percentile {np.percentile(best_lag, 25):.0f} to {np.percentile(best_lag, 75):.0f} s")
pk_own = own.max(1); pk_oth = oth.max(1)
print(f"peak |r|, own temperature {np.median(pk_own):.3f} against next plane's temperature {np.median(pk_oth):.3f} (medians); "
      f"own higher in {np.mean(pk_own > pk_oth):.1%} of planes")
by_fish = pd.DataFrame({"fish": [r["fish"].split("_")[0] for r in ok], "lag": best_lag, "own": pk_own, "other": pk_oth})
g = by_fish.groupby("fish").agg(median_peak_lag_s=("lag", "median"), own=("own", "median"), other=("other", "median"))
g["own_minus_other"] = g.own - g.other
print(g.round(3).to_string())
offs = md.CLOCK_OFFSETS * md.DT
HO = np.vstack([r["clock_ho_temp"] for r in ok]); HOm = np.vstack([r["clock_ho_temp_mean"] for r in ok])
HOo = np.array([r["clock_ho_temp_other"] for r in ok])
print("held-out correlation (top 5% of components) by displacement of the temperature trace (s):")
sel = np.isin(np.round(offs, 1), [-30, -20, -15, -12, -10.4, -10, -8, -6, -5, -4, -3, -2, -1, 0, 1, 2, 3, 5, 10, 15, 20, 30])
print(pd.DataFrame({"top5%": HO.mean(0), "all components": HOm.mean(0)}, index=np.round(offs, 1)).loc[sel].round(3).T.to_string())
best = offs[np.argmax(HO, 1)]
i0 = list(md.CLOCK_OFFSETS).index(0)
print(f"per-plane best displacement: 0 s in {np.mean(best == 0):.1%} of planes, within 2 s in {np.mean(np.abs(best) <= 2):.1%}; "
      f"median {np.median(best):.1f} s")
print(f"at 0 s, own temperature {np.median(HO[:, i0]):.3f} against next plane's {np.median(HOo):.3f} (medians over planes); "
      f"own higher in {np.mean(HO[:, i0] > HOo):.1%} of planes")
fb = pd.DataFrame({"fish": [r["fish"].split("_")[0] for r in ok], "best_s": best, "own_0": HO[:, i0], "other_0": HOo})
print(fb.groupby("fish").agg(planes=("best_s", "size"), best_is_0=("best_s", lambda x: np.mean(x == 0)), median_best_s=("best_s", "median"),
                             own_0=("own_0", "median"), other_0=("other_0", "median")).round(3).to_string())
okb = [r for r in ok if "clock_bout_top" in r]
lb = okb[0]["clock_bout_lags_s"]; B = np.vstack([r["clock_bout_top"] for r in okb])
print(f"swim starts: {len(okb)} planes with at least 3 bouts; mean top-5% r by lag (s):")
print(pd.Series(B.mean(0).round(3), index=lb).iloc[::5].to_string())
print(f"peak of the mean curve at {lb[np.argmax(B.mean(0))]:.1f} s")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(offs, HO.mean(0), "o-", label="own temperature"); ax[0].axhline(HOo.mean(), color="C1", label="next plane's, 0 s")
ax[0].set_xlabel("displacement of temperature trace (s)"); ax[0].set_ylabel("held-out r, top 5%"); ax[0].legend()
ax[1].plot(lb, B.mean(0)); ax[1].set_xlabel("lag of calcium after swim start (s)"); ax[1].set_ylabel("mean top-5% r")
plt.tight_layout(); plt.show()
print("axis ranges: left", HO.mean(0).min().round(3), "to", HO.mean(0).max().round(3), "; right", B.mean(0).min().round(3), "to", B.mean(0).max().round(3))

In [ ]:
# Clock alignment, as the round-1 verifier asked: the stored start times and rates, which end of the
# temperature trace carries its extra ~10.5 s, and the held-out curve at 0.2 s steps.
ck = pd.DataFrame([{**r["clocks"], "dandiset": r["dandiset"]} for r in res])
print("CaUnitRespones starting_time (s):", sorted(ck.ca_start.unique()), "; rate (Hz):", sorted(pp.ca_rate.unique()))
print("Temperature starting_time (s):", sorted(ck.temp_start.unique()), "; rate (Hz):", sorted(ck.temp_rate.unique()))
print("first tail timestamp (s), percentiles 0/50/100:", np.round(np.percentile(ck.tail_first, [0, 50, 100]), 3))
fine = (offs >= -15) & (offs <= 10)
mc = HO.mean(0)
print(f"held-out r (top 5%), mean over {len(ok)} planes, at 0.2 s steps from -15 to +10 s:")
for a in np.arange(-15, 10, 5):
    w = fine & (offs >= a - 1e-9) & (offs < a + 5 - 1e-9)
    print(f"  {a:+.0f} to {a + 4.8:+.1f} s:", " ".join(f"{v:.3f}" for v in mc[w]))
bf = offs[fine][np.argmax(HO[:, fine], 1)]
print(f"best displacement per plane within -15 to +10 s: median {np.median(bf):+.1f} s, 25th to 75th percentile "
      f"{np.percentile(bf, 25):+.1f} to {np.percentile(bf, 75):+.1f} s; within 2 s of 0 in {np.mean(np.abs(bf) <= 2):.1%}, "
      f"within 2 s of -10.4 s in {np.mean(np.abs(bf + 10.4) <= 2):.1%}")
i_s, i_e = int(np.argmin(np.abs(offs - 0))), int(np.argmin(np.abs(offs + 10.4)))
print(f"temperature starting with imaging (0 s): median r {np.median(HO[:, i_s]):.3f}; ending with imaging (-10.4 s): "
      f"{np.median(HO[:, i_e]):.3f}; the first higher in {np.mean(HO[:, i_s] > HO[:, i_e]):.1%} of planes")
for r in ok[:: max(1, len(ok) // 3)][:3]:
    h = r["clock_ho_temp"][fine]
    print(f"  {r['fish'].split('_')[0]} plane {r['plane']}: best {offs[fine][np.argmax(h)]:+.1f} s; r at -10/-5/-1/0/+1/+5 s:",
          " ".join(f"{h[np.argmin(np.abs(offs[fine] - v))]:.3f}" for v in (-10, -5, -1, 0, 1, 5)))
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(offs[fine], mc[fine]); ax.axvline(0, color="k", lw=0.5); ax.axvline(-10.4, color="C3", lw=0.5)
ax.set_xlabel("displacement of temperature (s); <0 advanced"); ax.set_ylabel("held-out r, top 5%"); plt.show()
print("axis range:", mc[fine].min().round(3), "to", mc[fine].max().round(3))


The held-out prediction from the plane's own temperature (median 0.656 at no displacement) is far
above the prediction from the next plane's temperature (0.393), in 98.5% of planes and in every
fish. The displacement does not wrap around (it did in round 1, which put a step into the last test
rows for every advance and produced a collapse to 0.198 at -1 s that this method cannot give, as the
round-1 verifier showed on synthetic data). With that fixed, the curve is smooth: it is highest for
the temperature delayed by 0 to +5 s (0.649 at 0 s, 0.663 at +3 s), falls slowly for advances
(0.604 at -5 s, 0.497 at -10 s) and for delays beyond +5 s. The per-plane best displacement has a
median of +3.4 s (25th to 75th percentile +1.8 to +4.8 s), as expected when calcium lags a slow
stimulus. Because the 10 s history covers lags of several seconds whatever the displacement, this
check cannot locate the clocks to better than a few seconds.

All series store a starting time of 0 s (calcium at 2.384 Hz, temperature at 20 Hz), and the tail
timestamps start a median 5.4 s before the first frame. The temperature trace runs about 10.5 s
longer than imaging. Read as starting with imaging, it predicts the calcium better than read as
ending with it (a displacement of -10.4 s), in 98.3% of planes (median 0.656 against 0.493), and only
0.3% of planes have their best displacement within 2 s of -10.4 s. So the extra 10.5 s lies at the
end of the trace, as the code assumes; this is shown to within a few seconds, not exactly.
The correlation of calcium with swim starts is already above its level at -5 s (0.086) a second before
the bout (0.108 at -1 s), rises at the bout (0.144 at 0 s) and peaks 1.8 s after it (0.212 at 2 s). Both
checks are consistent with the calcium, stimulus and tail sharing one clock, to within a few seconds.

### Bout detection

The dandisets hold no swim bouts. Bouts are re-derived from the tail (see the helper's
docstring). The smoothed absolute angular derivative has a noise floor of a few rad/s, because
the tracked segment angles move in steps of about 0.1 rad; the code's default threshold of
2 rad/s lies inside that noise, so the readings use 5 and 10 rad/s.

In [ ]:
nb = pd.DataFrame([{**r["n_bouts"], "dandiset": r["dandiset"]} for r in res if r["dandiset"] in md.ZEBRAFISH_WHOLE_BRAIN])
dur = 495.8
print("bouts per plane (and per second) under each reading, whole-brain planes:")
for k in md.BOUT_READINGS:
    print(f"  {k:26s} median {nb[k].median():7.0f}  ({nb[k].median()/dur:.3f}/s); 5th to 95th percentile {nb[k].quantile(.05):.0f} to {nb[k].quantile(.95):.0f}; planes with none {(nb[k]==0).sum()}")
with md.open_nwb(assets["000235"][0]["asset_id"]) as f:
    seg = f["processing/behavior/Raw Tail Behavior Data/Tail_Z5/data"][:]
deriv, sm = md.tail_metrics(seg)
print("segment-angle steps (rad):", np.unique(np.round(np.abs(seg[~np.isnan(seg)]), 4))[:5])
print("smoothed |d angle / dt| (rad/s), fish 01 plane 5, percentiles 50/90/99/99.5/99.9:", np.round(np.nanpercentile(sm, [50, 90, 99, 99.5, 99.9]), 2))
print("bouts in that plane by threshold (min 10 frames):", {thr: md.detect_bouts(sm, 10, thr).shape[0] for thr in [2, 3, 5, 8, 10, 20]})
mb = pd.DataFrame([r["n_bins_multi_bout"] for r in res if r["dandiset"] in md.ZEBRAFISH_WHOLE_BRAIN])
print("0.2 s bins holding two or more bout starts (vigor and direction keep the later bout's sum there, as the code's "
      "bin_time_discrete does), total over the 750 planes:", mb.sum().to_dict(), "; against bouts:", nb[list(md.BOUT_READINGS)].sum().to_dict())


## 2. Findings in the data

In [ ]:
# Frame rate: the files' imaging rate and frame duration against the Methods' 2.5 Hz.
print("imaging rates in all", len(pp), "planes (Hz):", sorted(pp.ca_rate.unique()))
print("frame duration in the experiment description (s):", sorted(pp.frame_duration.unique()))
print("1182 frames at 2.384 Hz:", 1182 / 2.384185791015625, "s;  at the paper's 2.5 Hz:", 1182 / 2.5, "s; paper's stimulus per plane: 495 s")

In [ ]:
# What is not in the files: no bouts or behavior predictors, no anatomy registration, no backfill annotation.
with md.open_nwb(assets["000238"][0]["asset_id"]) as f:
    print("000238 processing/ophys holds:", sorted({k.split("_Z")[0] for k in f["processing/ophys"].keys()}))
    print("000238 segmentation columns:", f["processing/ophys/ImageSegmentation/Segmentation_Z0"].attrs["colnames"])
    print("000238 reference images:", list(f["processing/ophys/ImageSegmentation/Segmentation_Z0/reference_images"].keys()))
    print("000238 behavior:", list(f["processing/behavior"].keys()), "; general:", list(f["general"].keys()))
    print("000238 genotype:", f["general/subject/genotype"][()])
with md.open_nwb(assets["000235"][0]["asset_id"]) as f:
    print("000235 processing modules:", list(f["processing"].keys()), "; behavior:", list(f["processing/behavior"].keys()))
    print("000235 segmentation columns:", f["processing/ophys/ImageSegmentation/Segmentation_Z0"].attrs["colnames"])
    print("000235 genotype:", f["general/subject/genotype"][()], "; location of plane 0:", f["general/optophysiology/Z0_0/location"][()])
red = pp[pp.dandiset == "000238"].movie_shape_red
print("000238 red-channel movies (Z<k>_1):", red.notna().sum(), "of", len(red), "planes; shapes", sorted({tuple(s) for s in red.dropna()}))


In [ ]:
# Planes that differ from the rest: frame counts, tail time ranges, temperature trace lengths and ranges.
print("files with fewer than 30 planes:", fish[fish.planes != 30][["dandiset", "fish", "planes"]].to_string(index=False))
print("planes with 1181 frames:", pp[pp.n_frames != 1182][["dandiset", "fish_short", "plane", "n_frames"]].to_string(index=False))
odd_tail = pp[(pp.tail_t0 < -10) | (pp.tail_t1 > 500)]
print("planes whose tail time starts before -10 s or ends after 500 s:")
print(odd_tail[["dandiset", "fish_short", "plane", "tail_n", "tail_t0", "tail_t1"]].to_string(index=False))
print("tail start time (s) over all planes, percentiles 1/50/99:", np.round(np.percentile(pp.tail_t0, [1, 50, 99]), 2),
      "; end:", np.round(np.percentile(pp.tail_t1, [1, 50, 99]), 2))
dur_t = pp.temp_n / 20; dur_c = pp.n_frames * pp.frame_duration
print(f"temperature trace minus imaging duration (s): median {np.median(dur_t - dur_c):.2f}, range {np.min(dur_t - dur_c):.2f} to {np.max(dur_t - dur_c):.2f}")
hot = pp[pp.temp_max > 30]
print(f"planes whose temperature exceeds 30 C (paper: 22 to 30 C): {len(hot)} of {len(pp)}; highest {pp.temp_max.max():.2f} C;",
      hot.groupby("fish_short").size().to_dict())
ck = pd.DataFrame([{**r["clocks"], "fish_short": r["fish"].split("_")[0].replace("sub-", ""), "plane": r["plane"], "dandiset": r["dandiset"]} for r in res])
odd = ck.merge(odd_tail[["dandiset", "fish_short", "plane"]])
print("in those planes, the largest departure of a tail time step from the median step:")
print(odd[["dandiset", "fish_short", "plane", "tail_max_step", "tail_max_step_at", "tail_max_step_index", "tail_steps_over_1s"]].round(3).to_string(index=False))
rest = ck[~ck.set_index(["dandiset", "fish_short", "plane"]).index.isin(odd.set_index(["dandiset", "fish_short", "plane"]).index)]
print(f"other planes: largest step {rest.tail_max_step.max():.4f} s; planes with a step over 1 s: {(rest.tail_steps_over_1s > 0).sum()}")

# Which side of the jump is misplaced: moving the side beyond it (after the jump in the planes ending at
# 623.8 s, before it in those starting at -133.5 s) by the jump gives the time range of every other plane.
fx = odd.merge(odd_tail[["dandiset", "fish_short", "plane", "tail_n", "tail_t0", "tail_t1"]])
late = fx.tail_t1 > 500
fx["side_moved"] = np.where(late, "after the jump", "before the jump")
fx["t0_moved"] = np.where(late, fx.tail_t0, fx.tail_t0 + fx.tail_max_step)
fx["t1_moved"] = np.where(late, fx.tail_t1 - fx.tail_max_step, fx.tail_t1)
print("with the misplaced side moved by the jump:")
print(fx[["dandiset", "fish_short", "plane", "side_moved", "t0_moved", "t1_moved", "tail_n"]].round(2).to_string(index=False))
rp = pp[~pp.set_index(["dandiset", "fish_short", "plane"]).index.isin(fx.set_index(["dandiset", "fish_short", "plane"]).index)]
print("other planes, percentiles 1/50/99: start", np.round(np.percentile(rp.tail_t0, [1, 50, 99]), 2), "; end",
      np.round(np.percentile(rp.tail_t1, [1, 50, 99]), 2), "; samples", np.percentile(rp.tail_n, [1, 50, 99]).astype(int))

The attempt record states which of these are findings.

## 3. data-zebrafish-coverage

> The zebrafish dataset holds 750 imaging planes from 25 larvae, with 706,543 CaImAn components
> of which 432,882 pass the brightness criterion.

Counts over 000235, 000236 and 000237. Brightness is the mean photon count of the GCaMP movie over
imaging time, averaged over the component's pixels (Methods; `avg_component_brightness` in the
code); components below 0.1 are excluded. The cut is also shown at other levels.

In [ ]:
wb = pp[pp.dandiset.isin(md.ZEBRAFISH_WHOLE_BRAIN)]
brs = np.concatenate([r["brightness"] for r in res if r["dandiset"] in md.ZEBRAFISH_WHOLE_BRAIN])
n_fish = wb.fish.nunique(); n_planes = len(wb); n_comp = int(wb.n_rois.sum()); n_bright = int(np.sum(brs >= md.BRIGHTNESS_CUT))
print(f"fish {n_fish} (expected 25), planes {n_planes} (expected 750)")
print(f"components {n_comp} (expected 706,543, tolerance 70,654): {n_comp/706543-1:+.2%}")
print(f"components with brightness >= 0.1: {n_bright} (expected 432,882, tolerance 43,288): {n_bright/432882-1:+.2%}; "
      f"fraction {n_bright/n_comp:.3f} (paper {432882/706543:.3f})")
print("NaN brightness (components with no pixels):", int(np.isnan(brs).sum()))
for cut in [0.05, 0.08, 0.09, 0.1, 0.11, 0.12, 0.15]:
    print(f"   cut {cut:.2f}: {int(np.sum(brs >= cut))}")
per_ds = wb.groupby("dandiset").agg(fish=("fish", "nunique"), planes=("plane", "size"), components=("n_rois", "sum"))
per_ds["bright"] = [int(np.sum(np.concatenate([r["brightness"] for r in res if r["dandiset"] == d]) >= 0.1)) for d in per_ds.index]
print(per_ds.to_string())
pf = pd.DataFrame([{"fish": r["fish"].split("_")[0], "n": r["n_rois"], "bright": int(np.sum(r["brightness"] >= 0.1))} for r in res if "brightness" in r]).groupby("fish").sum()
pf["fraction"] = (pf.bright / pf.n).round(3)
print(pf.T.to_string())

In [ ]:
in_tol = lambda v, e, t: abs(v - e) <= t
checks = {"fish": n_fish == 25, "planes": n_planes == 750, "components": in_tol(n_comp, 706543, 70654),
          "bright components": in_tol(n_bright, 432882, 43288)}
print(checks, "-> all within tolerance:", all(checks.values()))

**Result: reproduced.** The three dandisets hold 25 fish and 750 planes, as the paper states, and
706,543 CaImAn components, exactly the paper's count. 434,651 components have a mean brightness of
at least 0.1 photons, 0.41% more than the paper's 432,882 (a fraction of 0.615 against 0.613),
well within tolerance. The count moves by about 4,000 for each 0.01 of the cut (430,311 at 0.11,
439,048 at 0.09). The brightness is computed on the movie the files hold.

## 4. figs6-train-test-uncorrelated

> Because stimulus and behavior are random, test and training data are largely uncorrelated.

As `rwave_plot_panels.py`: each plane's temperature, swim-start, vigor and direction traces (raw,
before the Gram-Schmidt step) cut to the shortest plane length, split into three equal trials, and
the Pearson correlation of every pair of trials. Readings of "largely uncorrelated" (criterion): (a)
the median of each predictor's inter-trial correlations below 0.2 in absolute value, (b) below 0.1.
The median of the absolute correlations is shown as a stricter variant.
The behavior traces depend on the bout reading; all are shown. The temperature trace does not.

In [ ]:
wbres = [r for r in res if r["dandiset"] in md.ZEBRAFISH_WHOLE_BRAIN]
L = min(r["temp5"].size for r in wbres); tl = L // 3
trials = [slice(0, tl), slice(tl, 2 * tl), slice(2 * tl, 3 * tl)]
def inter_trial(x):
    out = []
    for j, a in enumerate(trials):
        for b in trials[j + 1:]:
            xa, xb = x[a], x[b]
            out.append(np.corrcoef(xa, xb)[0, 1] if xa.std() > 0 and xb.std() > 0 else np.nan)
    return out
rows = []
for reading in md.BOUT_READINGS:
    cs = {"Temperature": [], "Swim": [], "Vigor": [], "Directionality": []}
    for r in wbres:
        cs["Temperature"] += inter_trial(r["temp5"][:L])
        b = r["behavior"][reading][:, :L]
        cs["Swim"] += inter_trial(b[0]); cs["Vigor"] += inter_trial(b[1]); cs["Directionality"] += inter_trial(b[2])
    for k, v in cs.items():
        v = np.array(v)
        rows.append(dict(bouts=reading, predictor=k, n=int(np.isfinite(v).sum()), undefined=int(np.isnan(v).sum()),
                         median=np.nanmedian(v), median_abs=np.nanmedian(np.abs(v)), p5=np.nanpercentile(v, 5), p95=np.nanpercentile(v, 95),
                         min=np.nanmin(v), max=np.nanmax(v)))
    if reading == md.PRIMARY_BOUTS:
        main_cs = {k: np.array(v) for k, v in cs.items()}
tt = pd.DataFrame(rows)
print(f"{len(wbres)} planes, trial length {tl} bins of 0.2 s")
print(tt.round(3).to_string(index=False))
fig, ax = plt.subplots(figsize=(6, 3))
ax.boxplot([v[np.isfinite(v)] for v in main_cs.values()], tick_labels=list(main_cs.keys()))
ax.set_ylabel("inter-trial correlation"); ax.set_title(f"bouts {md.PRIMARY_BOUTS}"); plt.show()
print("axis range:", min(np.nanmin(v) for v in main_cs.values()).round(3), "to", max(np.nanmax(v) for v in main_cs.values()).round(3))

In [ ]:
# Per dandiset (each fish contributes 30 planes x 3 pairs), primary bout reading.
rows = []
for d in md.ZEBRAFISH_WHOLE_BRAIN:
    sub = [r for r in wbres if r["dandiset"] == d]
    for k, i in [("Temperature", None), ("Swim", 0), ("Vigor", 1), ("Directionality", 2)]:
        v = np.array(sum([inter_trial(r["temp5"][:L] if i is None else r["behavior"][md.PRIMARY_BOUTS][i, :L]) for r in sub], []))
        rows.append(dict(dandiset=d, predictor=k, median_abs=np.nanmedian(np.abs(v)), median=np.nanmedian(v)))
print(pd.DataFrame(rows).pivot(index="predictor", columns="dandiset", values="median_abs").round(3).to_string())
# The criterion: "the median of each predictor's inter-trial correlations below 0.2 in absolute value"
# (reading a) or below 0.1 (reading b), that is |median|. The median of |r| is shown as a stricter variant.
ma = tt.groupby("bouts").apply(lambda g: pd.Series({"max_abs_median": g["median"].abs().max(), "a_|median|<0.2": bool((g["median"].abs() < 0.2).all()),
                                                     "b_|median|<0.1": bool((g["median"].abs() < 0.1).all()), "variant_max_median_abs": g.median_abs.max(),
                                                     "variant_median|r|<0.2": bool((g.median_abs < 0.2).all()), "variant_median|r|<0.1": bool((g.median_abs < 0.1).all())}),
                               include_groups=False)
print(ma.to_string())

In [ ]:
# Variant: the behavior predictors as the CNN sees them (mean-subtracted, Gram-Schmidt orthogonalized and
# z-scored, rwave_data_fit.py). The paper's panel uses the raw ones: rwave_plot_panels.py reads
# "bout_start", "rolling_vigor" and "sum_tail" from rwave_build_main.get_behaviors, which reads the
# "raw_behavior" stored before the orthogonalization.
import mine_cnn
rows = []
for reading in (md.PRIMARY_BOUTS, "thr2_min10_code_default"):
    cs = {"Swim": [], "Vigor": [], "Directionality": []}
    for r in wbres:
        X = mine_cnn.prepare_predictors(r["temp5"], r["behavior"][reading])[:L]
        for k, i in zip(cs, (1, 2, 3)):
            cs[k] += inter_trial(X[:, i].astype(np.float64))
    for k, v in cs.items():
        v = np.array(v)
        rows.append(dict(bouts=reading, predictor=k, n=int(np.isfinite(v).sum()), median=np.nanmedian(v),
                         median_abs=np.nanmedian(np.abs(v)), p5=np.nanpercentile(v, 5), p95=np.nanpercentile(v, 95)))
print(pd.DataFrame(rows).round(3).to_string(index=False))


**Result: reproduced.** Over the 750 planes (2,250 trial pairs) the median inter-trial correlation
is -0.051 for temperature and between -0.009 and -0.001 for swim starts, vigor and direction,
under every bout reading, so all four are below 0.1 in absolute value (readings a and b both
hold). The behavior traces are uncorrelated by any measure (median absolute correlation 0.015 or
less under the main reading). The temperature correlations spread widely, from -0.909 to 0.873
(5th to 95th percentile -0.620 to 0.544), and their median absolute value is 0.266, so the
stricter variant (median of absolute values below 0.2) fails for temperature. Under the main
bout reading 117 of 2,250 behavior pairs are undefined, from trials without a bout. The behavior traces as the CNN sees them (orthogonalized and z-scored) give medians within
0.009 of zero too (median absolute values 0.025 or less), so whether the paper's panel used raw or
orthogonalized traces does not matter; the code uses the raw ones.

## 5. The fits: the linear comparison model on every component, and how long MINE takes here

The remaining zebrafish claims need, for every one of the 432,882 bright components, the MINE CNN
of the code (`get_standard_model`: 80 linear filters over 50 timepoints, two 64-unit swish layers,
dropout 0.5, L1 1e-3, Adam 1e-3, batch 256, 100 epochs on the first two-thirds of the plane), the
same again on rotated control data (figs6-shuffle-enrichment), and Taylor bootstraps on the fit
neurons. The linear comparison model is closed-form, so it is fit here to every bright component
in the plane pass (`mine_data.lm_test_r`, following rwave_data_fit.py and
create_time_shifted_predictor_matrix; checked against sklearn's Ridge to 1e-12 while preparing).
That gives the linear model's side of fig6d-mine-identifies-more-than-lm and
figs6-mine-more-at-every-cutoff; the MINE side is estimated from a random subsample in
`mine_fits_subsample.ipynb`. The second cell fits the paper's architecture to 20 bright components
of one plane, on one thread, to time it.


In [ ]:
lmr = {k: np.concatenate([r["lm_r"][k] for r in wbres if "lm_r" in r]) for k in md.BOUT_READINGS}
no_lm = [r for r in wbres if "lm_r" not in r]
print(f"planes without the linear model (fewer than 5 bright components): {len(no_lm)}, "
      f"holding {sum(int(np.sum(r['brightness'] >= md.BRIGHTNESS_CUT)) for r in no_lm)} bright components")
CUT = np.sqrt(0.5)
for k, v in lmr.items():
    v = v[np.isfinite(v)]
    print(f"{k:26s} bright components fit {v.size}; test r median {np.median(v):.3f}, 99th percentile {np.percentile(v, 99):.3f}, "
          f"max {v.max():.3f}; at least sqrt(0.5): {np.sum(v >= CUT)}; at least 0.6 (the code's printed count): {np.sum(v >= 0.6)}")
v = lmr[md.PRIMARY_BOUTS]
cuts = np.round(np.arange(0, 1.001, 0.05), 2)
print("linear-model count by test-correlation cutoff (main bout reading):")
print(pd.Series([int(np.sum(v >= c)) for c in cuts], index=cuts).to_string())
lm_fish = pd.DataFrame([dict(dandiset=r["dandiset"], fish=r["fish"].split("_")[0], n=int(np.sum(np.isfinite(r["lm_r"][md.PRIMARY_BOUTS]))),
                             lm=int(np.sum(r["lm_r"][md.PRIMARY_BOUTS] >= CUT))) for r in wbres if "lm_r" in r]).groupby(["dandiset", "fish"]).sum()
lm_fish["fraction"] = lm_fish.lm / lm_fish.n
print("per fish, linear model at sqrt(0.5): fraction of bright components from", lm_fish.fraction.min().round(4), "to",
      lm_fish.fraction.max().round(4), "; fish with none:", int((lm_fish.lm == 0).sum()))
print(lm_fish.groupby("dandiset")[["n", "lm"]].sum().to_string())
print("paper, for comparison: union of MINE and the linear model 42,229, of which 60% (about 25,300) by the linear model")


In [ ]:
import mine_cnn
r0 = next(r for r in res if r["dandiset"] == "000235" and r["plane"] == 5)
with md.open_nwb(r0["asset_id"]) as f:
    C = f["processing/ophys/CaUnitRespones_Z5/data"][:]
ca_t, i_t = md.time_base(C.shape[0], r0["frame_duration"])
bright = np.nonzero(r0["brightness"] >= 0.1)[0]
X = mine_cnn.prepare_predictors(r0["temp5"], r0["behavior"][md.PRIMARY_BOUTS])
seed = zlib.crc32(b"costabile-2023/fig6d-mine-identifies-more-than-lm")
times, rs = [], []
for k, j in enumerate(bright[:20]):
    y = mine_cnn.prepare_response(np.interp(i_t, ca_t, C[:, j]))
    t = time.time()
    r_test = mine_cnn.fit_and_test(X, y, seed + k)
    times.append(time.time() - t); rs.append(r_test)
print("CPUs:", os.cpu_count(), "; usable:", len(os.sched_getaffinity(0)))
print("fit times (s):", np.round(times, 2))
print("test correlations (timing fits only, they test nothing):", np.round(rs, 3))
per_fit = float(np.median(times[1:]))
print(f"median of fits 2 to 20: {per_fit:.2f} s on one thread; range {min(times[1:]):.2f} to {max(times[1:]):.2f} s")
n_fits = 432882
hours_1 = per_fit * n_fits / 3600
print(f"real data: {n_fits} fits x {per_fit:.2f} s = {hours_1:.0f} h on one thread, {hours_1/4:.0f} h on 4 if four processes kept that speed "
      f"(they do not: see mine_fits_subsample.ipynb); with the rotated controls {2*hours_1/4:.0f} h, before any Taylor bootstrap")
print(f"on 4 CPUs at this speed the real-data fits take {hours_1/4/5:.1f} times a five-hour run, {2*hours_1/4/5:.1f} times with the controls")
print("the backfill fish (000238):", int(pp[pp.dandiset == "000238"].n_rois.sum()), "components, about",
      round(per_fit * pp[pp.dandiset == "000238"].n_rois.sum() / 3600 / 4), "h on 4 threads without the brightness cut")
# The simulation claims (data elsewhere) rest on fits of the same network to 15,000-frame simulations
# (cnn_fit_test.py: n_frames 3000 * 5, two-thirds for training), so each fit sees about 6 times as
# many training samples as a plane here.
train_here = (2478 * 2) // 3 - 49
train_sim = (15000 * 2) // 3 - 49
per_sim_fit = per_fit * train_sim / train_here
fits = {"fig1f and fig4 (11 response types x 20)": 220, "fig2e and fig2s1 (500 sets x 4 responses)": 2000,
        "fig3 and fig3s1 white noise (2 filter sets x 100)": 200, "mouse, Figure 5 (13 sessions x 200 components)": 2600}
for k, n in fits.items():
    h = n * per_sim_fit / 3600
    print(f"{k}: {n} fits x {per_sim_fit:.0f} s = {h:.1f} h on one thread, {h/4:.1f} h on 4; x5 for the protocol's five seeds: {5*h/4:.1f} h")
print(f"notebook elapsed so far: {(time.time()-T0)/60:.1f} min")

**Result: the fits are too large for this run; the linear model's count is close to the paper's.**
The linear comparison model, fit to every one of the 434,648 bright components with enough
neighbours, identifies about 26,300 at a test correlation of sqrt(0.5) under the main bout reading (about
25,400 to 26,700 under the others; the exact counts are printed above and move by a few components
between runs, through small numerical differences near the cut). The paper's 42,229 neurons identified by either model, 60% of them by
the linear model (54% by both, 6% by it alone), imply about 25,300, so the linear model alone comes
within 4% of the paper's count. MINE's side is estimated from a random subsample in
`mine_fits_subsample.ipynb`. The time of one fit of the paper's network on one thread here, and
the hours the 432,882 real-data fits would take, are printed above (the median of fits 2 to 20; the
first includes building the graph). Even on four CPUs at that speed, which four parallel processes do
not keep (`mine_fits_subsample.ipynb`), the fits take many times the five hours of a run, and twice
as long with the rotated controls; the timing varies between runs by tens of percent, and the
printed comparison with the run's length is what decides the block. Every claim that rests
on the MINE fits is therefore blocked as too large, as is each claim that depends on them. The 20
test correlations printed are from the timing fits only and test nothing.


## 6. Summary

| Claim | Outcome |
|---|---|
| data-zebrafish-coverage | reproduced |
| figs6-train-test-uncorrelated | reproduced (rests on the bout-threshold assumption) |
| 12 claims on the CNN fits | blocked, too_large (the linear model's side of fig6d is computed in section 5; MINE's is estimated in `mine_fits_subsample.ipynb`) |
| 6 anatomical claims (figs6 regions, fig7a, fig7c, fig7f) | blocked, too_large through their dependence on the fits; they also need the Z-Brain registration, which is not in the dandisets |
| fig6f-reticulospinal-behavior | blocked, data_missing (no backfill annotation in 000238); it would also need the fits |
| 14 claims on simulations and mouse widefield data (data elsewhere) | blocked, other: not attempted in this run |

How this analysis differs from the paper: brightness is computed on the movie the files hold
(whether CaImAn's projection came from a motion-corrected movie is not stated); bouts are re-derived
from the tail with thresholds above the tracking noise, since the files hold no bout calls and the
paper gives no threshold.

What was found about the dataset: the counts of fish, planes and components match the paper
exactly, and the brightness cut gives the paper's count within 0.4%. No plane is deposited twice.
The calcium follows its own plane's temperature far better than the next plane's, and the swim
starts by about 2 s, which is consistent with shared clocks; the temperature trace's extra 10.5 s
lies at its end. In 10 planes the tail timestamps jump once by 128.004 s, so part of the tail
recording of each is misplaced by 128 s relative to the imaging; the time ranges show which part,
and moving it by the jump gives the range of every other plane. The files hold no swim bouts or
behavior predictors, no registration to the Z-Brain atlas, and no annotation of the backfilled
neurons, so most of the paper's analyses need these re-derived or taken from elsewhere. The
imaging rate is 2.384 Hz, where the Methods say 2.5 Hz. Section 2 lists the rest.
